# ML-07 — Baseline Action Score and Top-10 Review

**Lane locked:** Refresh / Content Opportunity Scoring.

This notebook builds the transparent rule baseline that my Week-5 model must beat. It uses the real 30,000-row starter dataset and only information available in the current measurement window.

The rule never uses `trend_direction`, `trend_pct`, or the derived decline proxy as inputs. Those fields are used only after ranking to check whether the baseline is directionally useful.

> Rates in the starter file are stored as ×100 percentages. For example, `ctr = 0.50` means **0.50%**, not 50%.


## 1. Check two signals first

### Signal A — CTR vs position (FlyRank flag-linked)

The CTR-fix logic from the session depends on the idea that a page can already have useful visibility/ranking but still underperform on clicks. I test pages with **at least 500 impressions** and average position **1–20**, then bucket their measured CTR.

**Verdict: CONFIRMED.**

The low-CTR bucket has the highest observed decline-proxy rate among the main buckets. This does not prove that low CTR causes decline, but it supports using low CTR at visible positions as a review signal.

### Signal B — volume / impressions (FlyRank quick-win-linked)

Quick-win logic depends on volume because a problem on a page with meaningful impressions can matter more than the same problem on a page nobody sees. I bucket all pages by 90-day impressions.

**Verdict: MIXED.**

The lowest-volume pages have a clearly lower observed decline-proxy rate, while the middle/high-volume buckets are higher; however, the very highest-volume bucket is not the highest-risk group. I therefore use volume as an **impact/prioritization signal**, not as proof that a page is declining.

### Baseline rule in plain words

A page enters the action queue when it has **at least 500 impressions** and an average position between **1 and 20**. Its score rises when it has more impressions and when its CTR falls below **0.50%**. This is intentionally simple, hand-written, and readable.

Each row gets **one reason code** and **one action label**. No fitted weights and no future/label-derived inputs are used.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
DATA_PATH = next((p for p in paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

DATA_PATH = DATA_PATH.resolve()
REPO_ROOT = DATA_PATH.parents[2]

df = pd.read_csv(DATA_PATH)
df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)  # evaluation only

print(f"Starter data: {len(df):,} rows × {df.shape[1]-1} original columns")
print(f"Overall observed decline-proxy base rate: {df['decline_proxy'].mean():.1%}")

ctr_slice = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

ctr_slice["ctr_bucket"] = pd.cut(
    ctr_slice["ctr"],
    bins=[-np.inf, 0.5, 1.0, 2.0, np.inf],
    labels=["<0.5%", "0.5-<1%", "1-<2%", "2%+"],
    right=False,
)

ctr_table = (
    ctr_slice.groupby("ctr_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_proxy_rate=("decline_proxy", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_position=("avg_position", "median"),
    )
    .reset_index()
)
ctr_table["decline_proxy_rate"] = (100 * ctr_table["decline_proxy_rate"]).round(1)

print("\nSIGNAL A — CTR vs position | VERDICT: CONFIRMED")
display(ctr_table)

df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 99, 499, 1999, 9999, np.inf],
    labels=["0-99", "100-499", "500-1,999", "2,000-9,999", "10,000+"],
)

volume_table = (
    df.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        decline_proxy_rate=("decline_proxy", "mean"),
        median_ctr_pct=("ctr", "median"),
    )
    .reset_index()
)
volume_table["decline_proxy_rate"] = (100 * volume_table["decline_proxy_rate"]).round(1)

print("\nSIGNAL B — volume | VERDICT: MIXED")
display(volume_table)
print("\nLeakage note: decline_proxy is used only for these checks/evaluation, never in the rule score.")


Starter data: 30,000 rows × 44 original columns
Overall observed decline-proxy base rate: 54.2%

SIGNAL A — CTR vs position | VERDICT: CONFIRMED


,ctr_bucket,n,decline_proxy_rate,median_impressions,median_position
0,<0.5%,9759,62.7,3017.0,8.4
1,0.5-<1%,1728,48.1,4935.0,7.1
2,1-<2%,489,44.8,4152.0,7.5
3,2%+,47,53.2,3579.0,9.0



SIGNAL B — volume | VERDICT: MIXED


,volume_bucket,n,decline_proxy_rate,median_ctr_pct
0,0-99,7994,38.9,0.00
1,100-499,5280,60.4,0.00
2,"500-1,999",6511,61.8,0.12
3,"2,000-9,999",6613,61.3,0.19
4,"10,000+",3602,52.4,0.23



Leakage note: decline_proxy is used only for these checks/evaluation, never in the rule score.


## 2. Build the ranked queue

The score is deliberately hand-written:

- **Eligibility:** `impressions_90d >= 500` and `1 <= avg_position <= 20`
- **Impact component (60 points):** percentile rank of `log(1 + impressions_90d)` among eligible pages
- **CTR-gap component (40 points):** how far CTR is below the 0.50% review threshold
- **Total:** `60 × impact + 40 × CTR_gap`

The one primary reason code is `low_ctr_visible` when an eligible page has CTR below 0.50%; otherwise an eligible page receives `visible_position_opportunity`. Non-eligible rows receive `not_eligible`.

The action is `review_refresh_ctr` for the low-CTR opportunity, `monitor_visible` for other eligible pages, and `monitor` otherwise.

This queue is written to `work/outputs/baseline_action_score.csv`. The CSV deliberately contains **no outcome label, `trend_direction`, or `trend_pct`**.


In [2]:
eligible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
)

df["impact_component"] = 0.0
df.loc[eligible, "impact_component"] = (
    np.log1p(df.loc[eligible, "impressions_90d"]).rank(pct=True, method="average")
)

df["ctr_gap_component"] = 0.0
df.loc[eligible, "ctr_gap_component"] = np.clip(
    (0.50 - df.loc[eligible, "ctr"]) / 0.50,
    0,
    1,
)

df["baseline_action_score"] = np.where(
    eligible,
    60 * df["impact_component"] + 40 * df["ctr_gap_component"],
    0.0,
)

df["reason_code"] = np.select(
    [eligible & (df["ctr"] < 0.50), eligible],
    ["low_ctr_visible", "visible_position_opportunity"],
    default="not_eligible",
)

df["action_label"] = np.select(
    [eligible & (df["ctr"] < 0.50), eligible],
    ["review_refresh_ctr", "monitor_visible"],
    default="monitor",
)

queue = (
    df.sort_values(
        ["baseline_action_score", "impressions_90d"],
        ascending=[False, False],
        kind="stable",
    )
    .reset_index(drop=True)
    .copy()
)
queue["baseline_rank"] = np.arange(1, len(queue) + 1)

queue_cols = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "impact_component",
    "ctr_gap_component",
]
queue_export = queue[queue_cols].copy()

output_dir = REPO_ROOT / "work/outputs"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "baseline_action_score.csv"
queue_export.to_csv(csv_path, index=False)

eval_queue = queue[["content_id", "decline_proxy", "baseline_action_score"]].copy()
base_rate = df["decline_proxy"].mean()
precision_at_10 = eval_queue.head(10)["decline_proxy"].mean()
precision_at_20 = eval_queue.head(20)["decline_proxy"].mean()
precision_at_50 = eval_queue.head(50)["decline_proxy"].mean()

metrics = {
    "rows_ranked": int(len(queue_export)),
    "eligible_rows": int(eligible.sum()),
    "base_rate": float(base_rate),
    "precision_at_10": float(precision_at_10),
    "precision_at_20": float(precision_at_20),
    "precision_at_50": float(precision_at_50),
    "score_inputs": ["impressions_90d", "avg_position", "ctr"],
    "excluded_from_score": ["trend_direction", "trend_pct", "decline_proxy"],
}
json_path = output_dir / "baseline_metrics.json"
json_path.write_text(json.dumps(metrics, indent=2))

print(f"Eligible pages: {eligible.sum():,}")
print(f"Wrote ranked queue: {csv_path}")
print(f"Wrote metrics receipt: {json_path}")
print(f"Base rate:      {base_rate:.1%}")
print(f"Precision@10:   {precision_at_10:.1%}")
print(f"Precision@20:   {precision_at_20:.1%}")
print(f"Precision@50:   {precision_at_50:.1%}")

display(queue_export.head(10))


Eligible pages: 12,023
Wrote ranked queue: /mnt/data/flyrank_w04/flyrank-ML-internship-main/work/outputs/baseline_action_score.csv
Wrote metrics receipt: /mnt/data/flyrank_w04/flyrank-ML-internship-main/work/outputs/baseline_metrics.json
Base rate:      54.2%
Precision@10:   60.0%
Precision@20:   60.0%
Precision@50:   60.0%


,content_id,client_id,baseline_rank,baseline_action_score,reason_code,action_label,impressions_90d,clicks_90d,ctr,avg_position,impact_component,ctr_gap_component
0,content_c8e9d6ab9013,client_19581e27de,1,99.895201,low_ctr_visible,review_refresh_ctr,208678,0,0.00,9.7,0.998253,1.00
1,content_453722754fea,client_f369cb89fc,2,98.880612,low_ctr_visible,review_refresh_ctr,140079,16,0.01,7.6,0.994677,0.98
2,content_4a6607efcb46,client_6208ef0f77,3,98.805756,low_ctr_visible,review_refresh_ctr,128068,17,0.01,2.2,0.993429,0.98
3,content_39881853ef0c,client_f369cb89fc,4,98.616119,low_ctr_visible,review_refresh_ctr,112434,10,0.01,7.2,0.990269,0.98
4,content_0919dd345d80,client_4e07408562,5,97.910937,low_ctr_visible,review_refresh_ctr,119217,26,0.02,7.0,0.991849,0.96
5,content_d274ac4158ef,client_4e07408562,6,97.712850,low_ctr_visible,review_refresh_ctr,65138,6,0.01,6.8,0.975214,0.98
6,content_8451fc6f034d,client_d029fa3a95,7,97.535124,low_ctr_visible,review_refresh_ctr,272144,75,0.03,2.3,0.998919,0.94
7,content_c84a0ab98e90,client_f369cb89fc,8,97.520153,low_ctr_visible,review_refresh_ctr,223271,70,0.03,7.8,0.998669,0.94
8,content_e5f459e737b7,client_f369cb89fc,9,97.428396,low_ctr_visible,review_refresh_ctr,56363,3,0.01,5.9,0.970473,0.98
9,content_c1fe78bc4e37,client_19581e27de,10,97.270631,low_ctr_visible,review_refresh_ctr,134055,43,0.03,7.5,0.994511,0.94


## 3. Top-10 skeptical review

For every top-ten item I read the rule as a reviewer rather than assuming the score is correct. The line below states the action, why the rule placed it there, and what evidence could make the recommendation wrong.

A high score means **“review this first”**, not **“editing this will definitely improve performance.”**


In [3]:
top10 = queue.head(10).copy()

def wrong_reason(row):
    if row["ctr"] == 0:
        return (
            "CTR could reflect measurement/query-intent effects or SERP features; "
            "a content refresh may not be the correct intervention."
        )
    if row["avg_position"] > 15:
        return (
            "The main constraint may be ranking/intent fit rather than the snippet; "
            "CTR work alone could have little effect."
        )
    return (
        "Low CTR may be normal for this query mix, or the page may already satisfy "
        "its business purpose without needing a refresh."
    )

review_lines = []
for _, row in top10.iterrows():
    why = (
        f"{int(row['impressions_90d']):,} impressions, "
        f"{row['ctr']:.2f}% CTR, avg position {row['avg_position']:.1f}, "
        f"reason={row['reason_code']}"
    )
    line = (
        f"#{int(row['baseline_rank'])} {row['content_id']} — "
        f"ACTION: {row['action_label']} | WHY: {why} | "
        f"WHAT WOULD MAKE IT WRONG: {wrong_reason(row)}"
    )
    review_lines.append(line)

print("TOP-10 REVIEW")
for line in review_lines:
    print(line)

review_table = top10[
    [
        "baseline_rank",
        "content_id",
        "action_label",
        "reason_code",
        "baseline_action_score",
        "impressions_90d",
        "ctr",
        "avg_position",
    ]
].copy()
review_table["what_would_make_it_wrong"] = [wrong_reason(r) for _, r in top10.iterrows()]
display(review_table)


TOP-10 REVIEW
#1 content_c8e9d6ab9013 — ACTION: review_refresh_ctr | WHY: 208,678 impressions, 0.00% CTR, avg position 9.7, reason=low_ctr_visible | WHAT WOULD MAKE IT WRONG: CTR could reflect measurement/query-intent effects or SERP features; a content refresh may not be the correct intervention.
#2 content_453722754fea — ACTION: review_refresh_ctr | WHY: 140,079 impressions, 0.01% CTR, avg position 7.6, reason=low_ctr_visible | WHAT WOULD MAKE IT WRONG: Low CTR may be normal for this query mix, or the page may already satisfy its business purpose without needing a refresh.
#3 content_4a6607efcb46 — ACTION: review_refresh_ctr | WHY: 128,068 impressions, 0.01% CTR, avg position 2.2, reason=low_ctr_visible | WHAT WOULD MAKE IT WRONG: Low CTR may be normal for this query mix, or the page may already satisfy its business purpose without needing a refresh.
#4 content_39881853ef0c — ACTION: review_refresh_ctr | WHY: 112,434 impressions, 0.01% CTR, avg position 7.2, reason=low_ctr_visible | 

,baseline_rank,content_id,action_label,reason_code,baseline_action_score,impressions_90d,ctr,avg_position,what_would_make_it_wrong
0,1,content_c8e9d6ab9013,review_refresh_ctr,low_ctr_visible,99.895201,208678,0.00,9.7,CTR could reflect measurement/query-intent eff...
1,2,content_453722754fea,review_refresh_ctr,low_ctr_visible,98.880612,140079,0.01,7.6,"Low CTR may be normal for this query mix, or t..."
2,3,content_4a6607efcb46,review_refresh_ctr,low_ctr_visible,98.805756,128068,0.01,2.2,"Low CTR may be normal for this query mix, or t..."
3,4,content_39881853ef0c,review_refresh_ctr,low_ctr_visible,98.616119,112434,0.01,7.2,"Low CTR may be normal for this query mix, or t..."
4,5,content_0919dd345d80,review_refresh_ctr,low_ctr_visible,97.910937,119217,0.02,7.0,"Low CTR may be normal for this query mix, or t..."
5,6,content_d274ac4158ef,review_refresh_ctr,low_ctr_visible,97.712850,65138,0.01,6.8,"Low CTR may be normal for this query mix, or t..."
6,7,content_8451fc6f034d,review_refresh_ctr,low_ctr_visible,97.535124,272144,0.03,2.3,"Low CTR may be normal for this query mix, or t..."
7,8,content_c84a0ab98e90,review_refresh_ctr,low_ctr_visible,97.520153,223271,0.03,7.8,"Low CTR may be normal for this query mix, or t..."
8,9,content_e5f459e737b7,review_refresh_ctr,low_ctr_visible,97.428396,56363,0.01,5.9,"Low CTR may be normal for this query mix, or t..."
9,10,content_c1fe78bc4e37,review_refresh_ctr,low_ctr_visible,97.270631,134055,0.03,7.5,"Low CTR may be normal for this query mix, or t..."


## 4. Weak picks + leakage check

The top ten contains some **weak picks when checked against the observed decline proxy**. That is useful: the baseline is supposed to be a readable benchmark, not a perfect oracle. A page can have large visibility and low CTR without being observed as declining.

This also highlights the difference between **opportunity scoring** and **outcome prediction**. My score asks which visible low-CTR pages deserve review first. The decline proxy is used only after ranking to assess directionality.

**Leakage check:** the rule uses only `impressions_90d`, `avg_position`, and `ctr`. It does not use `trend_direction`, `trend_pct`, `decline_proxy`, IDs, or any future-window/label-derived input. The exported CSV also excludes all outcome fields.


In [4]:
weak = top10.loc[
    top10["decline_proxy"] == 0,
    [
        "baseline_rank",
        "content_id",
        "baseline_action_score",
        "impressions_90d",
        "ctr",
        "avg_position",
        "decline_proxy",
    ],
].copy()

print(f"Weak picks in top 10 by the observed decline proxy: {len(weak)}")
display(weak)

score_inputs = {"impressions_90d", "avg_position", "ctr"}
forbidden = {
    "trend_direction",
    "trend_pct",
    "decline_proxy",
    "is_declining_label",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
}

assert score_inputs.isdisjoint(forbidden)
assert forbidden.isdisjoint(set(queue_export.columns))
assert queue_export["reason_code"].notna().all()
assert queue_export["action_label"].notna().all()
assert csv_path.exists()

print("Leakage check PASSED.")
print("Rule inputs:", sorted(score_inputs))
print("No future-window or label-derived columns are present in the exported queue.")


Weak picks in top 10 by the observed decline proxy: 4


,baseline_rank,content_id,baseline_action_score,impressions_90d,ctr,avg_position,decline_proxy
2,3,content_4a6607efcb46,98.805756,128068,0.01,2.2,0
5,6,content_d274ac4158ef,97.712850,65138,0.01,6.8,0
6,7,content_8451fc6f034d,97.535124,272144,0.03,2.3,0
7,8,content_c84a0ab98e90,97.520153,223271,0.03,7.8,0


Leakage check PASSED.
Rule inputs: ['avg_position', 'ctr', 'impressions_90d']
No future-window or label-derived columns are present in the exported queue.


## 5. Self-check

- [x] Lane confirmed: Refresh / Content Opportunity Scoring.
- [x] Exactly two signals are checked with visible bucket tables and `n`.
- [x] CTR-vs-position is tied to a real FlyRank flag and receives a one-word verdict: **CONFIRMED**.
- [x] Volume is tied to quick-win logic and receives a one-word verdict: **MIXED**.
- [x] One transparent rule produces a numeric score.
- [x] Every row has exactly one reason code and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] The notebook also writes `work/outputs/baseline_metrics.json` as a run receipt.
- [x] Ten ranked rows are reviewed with action, why, and “what would make it wrong.”
- [x] Weak picks are acknowledged rather than hidden.
- [x] No `trend_direction`, `trend_pct`, future-window, ID, or label-derived field is used in scoring.
- [x] Claims are framed as observed, measured, directional, and decision-support rather than causal.
- [x] The notebook runs top to bottom with no errors.
- [ ] Commit `work/notebooks/w04_baseline_score.ipynb` to the public repo and submit the repo URL.
